# Context Model Training from Raw .osz

This notebook trains the context model from raw `.osz` files and stores checkpoints in the repository.
It also configures W&B to avoid notebook-name and interactive login prompts.

In [ ]:
import os
from pathlib import Path

from src.model.train_cli import main as train_main

In [ ]:
# Fixed defaults requested
raw_osz_dir = Path("E:/batchbeatmapdownloadtesttemp")
data_root = Path("E:/batchbeatmapdownloadtest")
repo_root = Path.cwd()
checkpoint_dir = repo_root / "checkpoints" / "context"

epochs = 10
architecture_name = "taiko_context_transformer"
device = "cuda"

# Resume / logging toggles
use_resume_if_available = True
use_wandb = False
wandb_run_name = "test_10epochs"
wandb_log_every_batches = 100
wandb_offline = False

# W&B inline config (no environment setup needed)
wandb_api_key = "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"
wandb_notebook_name = "train_context_raw_data.ipynb"

print(f"raw_osz_dir={raw_osz_dir}")
print(f"data_root={data_root}")
print(f"checkpoint_dir={checkpoint_dir}")

In [ ]:
if use_wandb and not wandb_offline and not wandb_api_key.strip():
    raise RuntimeError(
        "wandb_api_key is empty while use_wandb=True and wandb_offline=False."
    )

print(f"use_wandb={use_wandb}")
print(f"wandb_offline={wandb_offline}")
print(f"wandb_notebook_name={wandb_notebook_name}")

In [ ]:
checkpoint_dir.mkdir(parents=True, exist_ok=True)
last_ckpt = checkpoint_dir / "last.ckpt"

argv = [
    str(raw_osz_dir),
    "--data-root", str(data_root),
    "--checkpoints-dir", str(checkpoint_dir),
    "--architecture-name", architecture_name,
    "--epochs", str(epochs),
    "--device", device,
]

if use_resume_if_available and last_ckpt.exists():
    argv.extend(["--resume-checkpoint", str(last_ckpt)])

if use_wandb:
    argv.extend([
        "--wandb",
        "--wandb-notebook-name", wandb_notebook_name,
        "--wandb-run-name", wandb_run_name,
        "--wandb-log-every-batches", str(wandb_log_every_batches),
    ])
    if wandb_api_key.strip():
        argv.extend(["--wandb-api-key", wandb_api_key])
    if wandb_offline:
        argv.append("--wandb-offline")

print("Running train_cli with args:")
print(argv)
rc = train_main(argv)
print(f"train_main returned {rc}")